# Lecture 8.6 — Tool Auditing and Argument Validation with `before_tool_callback`

**Design Pattern:** Request / Response Modification (P5) applied at the tool layer
**Callback used:** `before_tool_callback`
**Applied to:** `accountant_agent` (on the `sum_costs` tool)

In this lecture we add a `before_tool_callback` to `accountant_agent`. A single callback delivers three capabilities in sequence:

1. **Audit trail** — logs every `sum_costs` invocation with the exact cost list and the loop iteration number, producing a clean execution record showing exactly what the LLM submitted to the tool on each pass through the `LoopAgent`

2. **Argument validation and sanitization** — if the LLM accidentally passes a negative number or a non-numeric value in the costs list, the callback silently removes or corrects it before the tool runs, preventing a silent wrong total or a runtime type error

3. **Suspiciously-high cost flagging** — if any single cost in the list exceeds a configurable threshold (e.g. `$10,000`), the callback emits a warning log line flagging which item triggered it, then allows execution to proceed — argument *modification awareness* without blocking

---
**Changes from Lecture 8.5 (the complete diff):**
1. New constant: `HIGH_COST_THRESHOLD = 10_000`
2. New function: `audit_and_validate_sum_costs` — audit logging, sanitization, and high-cost flagging
3. One new keyword argument on `accountant_agent`: `before_tool_callback=audit_and_validate_sum_costs`
4. One new standalone demo cell — calls the callback directly with mock objects for all three scenarios

---
**Key distinction from 8.4:** This callback never blocks execution. It always returns `None`. The lesson is that `before_tool_callback` is not only for skipping — it is equally valuable purely for observation and argument mutation before proceeding.

---
**Expected output — clean inputs:**
```
[AUDIT] sum_costs called | agent: accountant_agent | iteration: 1
[AUDIT] costs submitted: [8000.0, 4500.0]
[AUDIT] All values valid. Proceeding.
```
**Expected output — invalid values sanitized:**
```
[VALIDATE] Removed invalid value: -500.0 (negative)
[VALIDATE] Removed invalid value: 'unknown' (non-numeric)
[AUDIT] Sanitized costs: [8000.0]
```
**Expected output — high cost flagged:**
```
[FLAG] Suspiciously high cost detected: 95000.0 (threshold: 50000)
[AUDIT] Proceeding with flagged costs.
```


## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [ ]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [ ]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

## 🤖 3. Model Configuration

Define the model names once here. To upgrade to a newer model in the future,
change these two constants — nothing else in the notebook needs to touch.

In [ ]:
# ── Model Configuration ───────────────────────────────────────────────────────
# Change these two constants to swap models across the entire notebook.
# To upgrade to a newer model in the future, update AGENT_MODEL and JUDGE_MODEL here.

AGENT_MODEL = "gemini-2.5-flash"  # used by all workflow agents
JUDGE_MODEL = "gemini-2.5-flash"   # used by the safety judge


## 🪝 3. [CARRIED OVER from 8.3, 8.4 & 8.5] Define the Observability Callbacks

These two callbacks are unchanged from Lecture 8.3 and carried forward through 8.4 and 8.5.
They fire at the **agent** boundary (entry and exit).

The new `audit_and_validate_sum_costs` in the next cells fires at the **tool** boundary — a different hook that intercepts the tool call before the tool function executes.

All four callbacks coexist and fire simultaneously during a live run:
- `guardrail_before_workflow` — `before_agent_callback` on `budget_optimizer_workflow` (agent boundary)
- `log_agent_entry` — `before_agent_callback` on `spending_proposer_agent` (agent boundary)
- `sanitize_cost_cutter_response` — `after_model_callback` on `cost_cutter_agent` (model boundary)
- `audit_and_validate_sum_costs` — `before_tool_callback` on `accountant_agent` (tool boundary) ← **8.6 NEW**
- `log_agent_exit` — `after_agent_callback` on `plan_retriever_agent` (agent boundary)


In [ ]:
from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

## 🛡️ 4. [CARRIED OVER from 8.4] Define the LLM-as-Judge Guardrail

This cell is **unchanged from Lecture 8.4**. The guardrail fires at the **agent** boundary on `budget_optimizer_workflow` — before any sub-agent starts.

### Why it is still here

The 8.6 tool callback (`before_tool_callback`) operates at the **tool** boundary — a completely different level. All callbacks coexist without interference:

| Callback | Hook | Fires on | Boundary |
|---|---|---|---|
| `guardrail_before_workflow` | `before_agent_callback` | `budget_optimizer_workflow` | Agent |
| `log_agent_entry` | `before_agent_callback` | `spending_proposer_agent` | Agent |
| `sanitize_cost_cutter_response` | `after_model_callback` | `cost_cutter_agent` | Model |
| `audit_and_validate_sum_costs` | `before_tool_callback` | `accountant_agent` | Tool |
| `log_agent_exit` | `after_agent_callback` | `plan_retriever_agent` | Agent |

### Return value contract (reminder)
- Return `None` → topic is safe, workflow runs normally
- Return `Content` → entire workflow cancelled instantly — no sub-agent ever starts


In [ ]:
# ============================================================
# Lecture 8.4 — LLM-as-Judge Guardrail  [CARRIED OVER — UNCHANGED]
# Design Patterns: Guardrails & Policy Enforcement (P1)
#                  Conditional Skipping of Steps (P6)
# ============================================================

from google.genai import client as genai_client

# Initialise a direct Gemini client for the safety judge.
# This is a raw API call — completely separate from the ADK runner.
safety_client = genai_client.Client()

# ── Judge prompts ─────────────────────────────────────────────────────────
# Two-stage design:
#   Prompt 1 — binary verdict (YES/NO). Fast, cheap, used on every request.
#   Prompt 2 — human-readable explanation. Only called when verdict is YES,
#              so clean topics pay no extra cost.

SAFETY_VERDICT_PROMPT = """
You are a strict safety officer for an event planning company.
Evaluate the following event planning request.

Does it involve any of the following:
- Dangerous or high-risk activities
- Weapons, arms, or military equipment
- Illegal substances or narcotics
- Illegal activities of any kind
- Activities that cannot be commercially insured
- Adult-only or explicit content
- Anything that exposes the company to legal or reputational risk

Reply with EXACTLY one word — either YES or NO.
No explanation. No punctuation. Just the single word.

Event request: "{topic}"
"""

SAFETY_REASON_PROMPT = """
You are a polite but firm safety officer for an event planning company.
A client has requested help planning an event, but it has been flagged as
unsafe or inappropriate for our business.

Write a short, professional refusal message (2-3 sentences) addressed to
the client. Explain specifically why this type of event falls outside what
the company can assist with. Be clear but courteous. Do not offer
workarounds or alternatives.

Event request: "{topic}"
"""


def guardrail_before_workflow(
    callback_context: CallbackContext,
) -> Optional[types.Content]:
    """
    before_agent_callback on budget_optimizer_workflow (the SequentialAgent).

    Two-stage LLM-as-judge:
      Stage 1 — fast binary verdict (YES/NO) on every request.
      Stage 2 — rich refusal explanation, only when Stage 1 says YES.

    The explanation is written into state["refusal_reason"] so the runner
    can surface it as the final response instead of "No plan found."

    Placed on the SequentialAgent so it fires once before ANY sub-agent
    starts. Returning Content cancels the entire workflow instantly.
    """
    topic = callback_context.state.get("topic", "")

    print("\n" + "─" * 60)
    print(f"[SAFETY JUDGE] Evaluating topic: '{topic}'")

    # ── Stage 1: Binary verdict ───────────────────────────────────────────
    verdict_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_VERDICT_PROMPT.format(topic=topic),
    )
    verdict = verdict_response.text.strip().upper()
    print(f"[SAFETY JUDGE] Verdict         : {verdict}")

    if "YES" not in verdict:
        print(f"  └─ ✅ SAFE — starting workflow.")
        print("─" * 60)
        return None                        # topic is safe, proceed normally

    # ── Stage 2: Rich explanation (only reached when blocked) ─────────────
    print(f"[SAFETY JUDGE] Generating refusal explanation...")
    reason_response = safety_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=SAFETY_REASON_PROMPT.format(topic=topic),
    )
    refusal_reason = reason_response.text.strip()
    print(f"[SAFETY JUDGE] Reason          : {refusal_reason}")
    print(f"  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.")
    print("─" * 60)

    # Write the rich explanation into session state.
    # The runner reads state["refusal_reason"] when state["final_presentation"]
    # is absent — this is how the final response reaches the user.
    callback_context.state["refusal_reason"] = refusal_reason

    # Return Content to cancel the entire workflow.
    return types.Content(
        role="model",
        parts=[types.Part(text=refusal_reason)],
    )


## 🧹 5. [CARRIED OVER from 8.5] Define the Response Sanitization Callback

This cell is **unchanged from Lecture 8.5**. The `after_model_callback` on `cost_cutter_agent` strips markdown fences and coerces string costs to floats before the framework processes the LLM response.

It fires at the **model** boundary — after the LLM responds, before the ADK processes the output further. The new 8.6 callback fires at the **tool** boundary — a later, narrower interception point.

### Why both are needed

| Callback | What it fixes | When it fires |
|---|---|---|
| `sanitize_cost_cutter_response` | Fence-wrapped JSON, string costs in plan | After `cost_cutter_agent` LLM responds |
| `audit_and_validate_sum_costs` | Negative values, non-numeric values in tool args | Before `sum_costs` executes |

These are complementary, not redundant. The sanitizer ensures the plan JSON written to state is clean. The auditor ensures the cost list extracted from that plan and sent to `sum_costs` is valid — regardless of where those values originally came from.


In [ ]:
# ============================================================
# Lecture 8.5 — Response Sanitization: after_model_callback
# Design Pattern: Request / Response Modification (P5)
# ============================================================

import re
from google.adk.models import LlmResponse
from google.genai import types as genai_types


def sanitize_cost_cutter_response(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> Optional[LlmResponse]:
    """
    after_model_callback for cost_cutter_agent.

    Intercepts the raw LLM response and fixes two consistent formatting flaws:
      1. Markdown code fences   -- ```json ... ```  wrapping the JSON
      2. String costs           -- "cost": "5000" instead of "cost": 5000

    Return value contract:
      - Function call response       -> return None immediately (don't touch tool calls)
      - Fences or string costs found -> return a NEW LlmResponse with cleaned content
      - Already clean               -> return None (original passes through unchanged)

    Why return a new object rather than mutating the original?
    Other callbacks in the chain may hold references to the original LlmResponse.
    Mutating it in-place would silently affect those other callbacks.
    Always build a new one.
    """
    agent_name = callback_context.agent_name

    # -- Guard: only process text responses ---------------------------------
    # LLM responses can contain function calls, not just text.
    # Check before assuming parts[0] is a text part.
    if (
        not llm_response.content
        or not llm_response.content.parts
        or llm_response.content.parts[0].function_call is not None
    ):
        return None   # Tool call — pass through untouched

    raw_text = llm_response.content.parts[0].text
    if not raw_text:
        return None

    cleaned = raw_text
    modified = False
    coerce_count = 0

    # -- Fix 1: Strip markdown code fences ----------------------------------
    fence_pattern = r"^\s*```(?:json)?\s*\n?(.*?)\n?\s*```\s*$"
    fence_match = re.search(fence_pattern, cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
        modified = True
        print(f"[SANITIZE] Stripped markdown fences from {agent_name} response")

    # -- Fix 2: Coerce string costs to float --------------------------------
    # Walk the parsed JSON and convert any value whose key contains 'cost'
    # from a string to a float.
    try:
        data = json.loads(cleaned)

        def coerce_costs(obj):
            nonlocal coerce_count
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if "cost" in key.lower() and isinstance(value, str):
                        try:
                            obj[key] = float(value)
                            coerce_count += 1
                        except ValueError:
                            pass   # leave non-numeric strings alone
                    else:
                        coerce_costs(value)
            elif isinstance(obj, list):
                for item in obj:
                    coerce_costs(item)

        coerce_costs(data)

        if coerce_count > 0:
            cleaned = json.dumps(data)
            modified = True
            print(f"[SANITIZE] Coerced {coerce_count} string cost(s) to float in {agent_name} response")

    except (json.JSONDecodeError, TypeError):
        # Not valid JSON — skip coercion, but still return cleaned text
        # if fences were stripped
        pass

    # -- Return -------------------------------------------------------------
    if not modified:
        print(f"[SANITIZE] Response already clean. Passing through.")
        return None   # No change needed — return None so original is used

    # Build a NEW LlmResponse — never mutate the original
    cleaned_content = genai_types.Content(
        role=llm_response.content.role,
        parts=[genai_types.Part(text=cleaned)],
    )
    return LlmResponse(content=cleaned_content)

## 🔍 5b. [NEW] High-Cost Threshold Constant

A single module-level constant controls the suspiciously-high-cost threshold. Defining it here makes it easy to adjust without hunting through the callback body.


In [ ]:
# ── 8.6 New Constant ──────────────────────────────────────────────────────────
# Costs above this value trigger a [FLAG] warning log line in the auditor.
# The callback still proceeds — this is observation-only, not blocking.

HIGH_COST_THRESHOLD = 10_000   # configurable — set low enough to flag luxury venue/catering costs in the workflow


## 🛠️ 5c. [NEW] Define the Tool Auditing and Validation Callback

This is the **only new callback in this lecture**.

### Why `before_tool_callback`?

The `before_tool_callback` fires **after** the LLM has decided to call a tool and produced the tool arguments, but **before** the tool function actually executes. This gives us a precise window to:

- **Observe** exactly what argument values the LLM generated
- **Mutate** those arguments in-place before the tool sees them
- **Flag** suspicious values without blocking execution

### Why `accountant_agent` and `sum_costs`?

`sum_costs` is called once per loop iteration — making it the perfect tool to attach an audit trail to. Students can watch the cost list change across iterations as `cost_cutter_agent` finds cheaper alternatives. It is also the tool whose correctness matters most: a wrong total from `sum_costs` causes `accountant_agent` to emit a wrong critique, which cascades into incorrect loop termination.

### New context type: `ToolContext`

This callback receives a `ToolContext` — distinct from the `CallbackContext` used in all previous callbacks. `ToolContext` exposes:
- `tool_context.agent_name` — which agent triggered the tool call
- `tool_context.state` — session state (read/write)
- `tool_context.actions` — ADK flow control (same as in tool functions)

### Callback signature

```python
def audit_and_validate_sum_costs(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext
) -> Optional[Dict]:
```

Notice: **three arguments** instead of two. `tool` gives us the tool object (for name checking). `args` is the mutable dict of arguments the LLM produced. `tool_context` gives us the execution context.

### Return value contract

| Condition | Return value | What happens |
|---|---|---|
| Wrong tool name | `None` immediately | Defensive guard — skip |
| All values valid | `None` | Tool runs with original args |
| Invalid values removed | `None` (args mutated in-place) | Tool runs with cleaned args |
| High cost detected | `None` (after warning log) | Tool runs — no blocking |

**Key insight:** This callback never blocks. Every path returns `None`. The tool always executes. The value is in the log trail and the silent argument sanitization — not in gating.

### Mutating `args` in-place

Unlike the `after_model_callback` where we built a **new** `LlmResponse`, here we mutate `args['costs']` directly. The framework reads the same dict after the callback returns — so in-place mutation is the correct pattern here. We still return `None` to signal "proceed with whatever is now in args".


In [ ]:
# ============================================================
# Lecture 8.6 — Tool Auditing and Argument Validation
# Design Pattern: Request / Response Modification (P5) at the tool layer
# ============================================================

from typing import Any, Dict
from google.adk.tools import ToolContext
from google.adk.tools.base_tool import BaseTool

# Module-level iteration counter — tracks which loop pass triggered the tool call.
# Reset to 0 each time you run a fresh workflow.
_sum_costs_iteration: int = 0


def audit_and_validate_sum_costs(
    tool: BaseTool,
    args: Dict[str, Any],
    tool_context: ToolContext,
) -> Optional[Dict]:
    """
    before_tool_callback for accountant_agent, applied to sum_costs.

    Delivers three capabilities in a single pass:
      1. Audit trail   — logs every invocation with costs list and iteration number
      2. Validation    — silently removes negative or non-numeric values before tool runs
      3. High-cost flag — warns when any single cost exceeds HIGH_COST_THRESHOLD

    Return value contract:
      Always returns None — this callback never blocks execution.
      When invalid values are found, args['costs'] is mutated in-place.
      The tool then receives the cleaned argument dict automatically.

    Why in-place mutation (not a new dict)?
      The framework passes the same args dict to the tool after the callback returns.
      Mutating it directly is the correct pattern here — unlike after_model_callback
      where we had to build a new LlmResponse object.

    Why check tool.name?
      A single before_tool_callback fires for EVERY tool call on the agent.
      accountant_agent only has sum_costs, but defensive tool.name checks future-proof
      the callback if additional tools are ever added to this agent.
    """
    global _sum_costs_iteration

    # -- Defensive guard: only act on sum_costs ---------------------------


    # -- 1. Audit trail ---------------------------------------------------
    print(f"[AUDIT] sum_costs called | agent: {agent_name} | iteration: {_sum_costs_iteration}")
    print(f"[AUDIT] costs submitted: {costs}")

    # -- 2. Argument validation and sanitization --------------------------


    # -- 3. High-cost flagging --------------------------------------------


    # Always return None — never block tool execution



## 🛠️ 6. Define Workflow Tools

**One change from Lecture 8.5** — highlighted with `# <- 8.6 CHANGE`:

- `sum_costs` — gains `before_tool_callback` support via `accountant_agent` (the callback is attached to the agent, not the tool itself)
- `exit_loop` — unchanged

The tools themselves are not modified. The callback intercepts the tool *call*, not the tool *definition*.


In [ ]:
import json
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 7. Create Tool Wrappers

Unchanged from Lecture 8.5.


In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model=AGENT_MODEL,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 8. Create Agents

**One change from Lecture 8.5** — highlighted with `# <- 8.6 CHANGE`:

- `accountant_agent` — gains `before_tool_callback=audit_and_validate_sum_costs`
- `spending_proposer_agent`, `cost_cutter_agent`, `plan_retriever_agent` — all unchanged from 8.5

Five callbacks now fire across the workflow simultaneously:
- `guardrail_before_workflow` — on `budget_optimizer_workflow` (before_agent_callback)
- `log_agent_entry` — on `spending_proposer_agent` (before_agent_callback)
- `sanitize_cost_cutter_response` — on `cost_cutter_agent` (after_model_callback)
- `audit_and_validate_sum_costs` — on `accountant_agent` (before_tool_callback) ← **8.6 CHANGE**
- `log_agent_exit` — on `plan_retriever_agent` (after_agent_callback)


In [ ]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# Guardrail has moved to the SequentialAgent — this agent is now clean.
# before_agent_callback=log_agent_entry carried over from 8.3 unchanged.
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model=AGENT_MODEL,
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,   # ← 8.3/8.4 (unchanged)
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# before_tool_callback added in 8.6 — audits and validates sum_costs arguments.
accountant_agent = Agent(
    name="accountant_agent",
    model=AGENT_MODEL,
    tools=[sum_costs],
    before_tool_callback=audit_and_validate_sum_costs,   # <- 8.6 CHANGE
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique",
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# after_model_callback from 8.5 — unchanged.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=AGENT_MODEL,
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan",
    after_model_callback=sanitize_cost_cutter_response,   # <- 8.5 CHANGE
)

# Agent 4: Presents the final approved plan (runs once after loop).
# Unchanged from 8.4.
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model=AGENT_MODEL,
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    Your output must be the content of the final plan presented in a clear and easy-to-read format.
    """,
    tools=[],
    output_key="final_presentation",
    after_agent_callback=log_agent_exit,   # ← 8.3/8.4 (unchanged)
)


## 🔄 9. Assemble the Loop and Sequential Workflows

Unchanged from Lecture 8.5. The new callback lives on `accountant_agent` inside the loop — no changes needed here.


In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent

# Unchanged from Section 5.
budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

# ← 8.4 (unchanged): guardrail still lives on the SequentialAgent.
# 8.5 change is on cost_cutter_agent, not here.
budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent],
    before_agent_callback=guardrail_before_workflow,   # ← 8.4 (unchanged)
)


## 🚀 10. Build the Execution Engine

Unchanged from Lecture 8.5. The auditing callback fires automatically inside the loop — the runner does not need to know about it.


In [ ]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={"budget": budget, "topic": topic, "COMPLETION_PHRASE": COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # Read the final response from session state.
    #
    # Two possible paths through the workflow:
    #
    #   ✅ CLEAN topic  → plan_retriever_agent runs and writes final_presentation.
    #                     We read that.
    #
    #   🚫 BLOCKED topic → guardrail cancels the workflow and writes refusal_reason
    #                      into state. plan_retriever never runs, so
    #                      final_presentation is never written. We read
    #                      refusal_reason instead.
    #
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state = final_session.state

    if "final_presentation" in state:
        final_response = state["final_presentation"]
    elif "refusal_reason" in state:
        final_response = state["refusal_reason"]
    else:
        final_response = "No response was generated."

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response


## ✨ 11. Initialize Session Service

Unchanged from Lecture 8.5.


In [ ]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

## 🔬 12. Standalone Callback Demo

Before running the full workflow, let's verify the callback works correctly by calling it directly with mock inputs.

This cell creates three mock scenarios — one with clean inputs, one with invalid values, one with a suspiciously high cost — and calls `audit_and_validate_sum_costs` on each.

No runner, no session, no agents required. We also reset the iteration counter so the demo starts cleanly.

**Scenario 1** — Clean inputs, normal run
**Scenario 2** — Invalid value sanitization (negative + non-numeric)
**Scenario 3** — High cost flag (above threshold, execution still proceeds)


In [ ]:
# ============================================================
# Standalone demo — call the callback directly with mock data
# ============================================================
from unittest.mock import MagicMock

# Reset the iteration counter before the demo
_sum_costs_iteration = 0

def make_mock_tool(name):
    """Build a minimal mock BaseTool with just a name attribute."""
    t = MagicMock()
    t.name = name
    return t

def make_mock_tool_context(agent_name):
    """Build a minimal mock ToolContext with just an agent_name."""
    ctx = MagicMock()
    ctx.agent_name = agent_name
    return ctx

mock_tool = make_mock_tool("sum_costs")
mock_ctx  = make_mock_tool_context("accountant_agent")

print("=" * 60)
print("SCENARIO 1 — Clean inputs, normal run")
print("=" * 60)
args1 = {"costs": [8000.0, 4500.0]}
audit_and_validate_sum_costs(mock_tool, args1, mock_ctx)
print(f"→ args after callback: {args1}")

print()
print("=" * 60)
print("SCENARIO 2 — Invalid value sanitization")
print("=" * 60)
args2 = {"costs": [8000.0, -500.0, "unknown"]}
audit_and_validate_sum_costs(mock_tool, args2, mock_ctx)
print(f"→ args after callback: {args2}")

print()
print("=" * 60)
print("SCENARIO 3 — Suspiciously high cost flagged")
print("=" * 60)
args3 = {"costs": [95000.0, 4500.0]}
audit_and_validate_sum_costs(mock_tool, args3, mock_ctx)
print(f"→ args after callback: {args3}")


## ▶️ 13a. Run — CLEAN Topic (Full Workflow with Auditing)

The topic `"50 person AI event in New York"` passes the guardrail and runs the full workflow.

Watch for `[AUDIT]` log lines appearing **on every loop iteration** as `accountant_agent` calls `sum_costs` on each pass. You will also see the existing `[SANITIZE]` lines from the 8.5 callback still firing.

You will see all five callback layers active simultaneously:
- `[SAFETY JUDGE]` — guardrail evaluates the topic
- `[ENTRY]` — `log_agent_entry` fires as `spending_proposer_agent` starts
- `[SANITIZE]` — `sanitize_cost_cutter_response` fires after each `cost_cutter_agent` LLM call
- `[AUDIT]` — `audit_and_validate_sum_costs` fires before each `sum_costs` tool call  ← **NEW**
- `[EXIT]` — `log_agent_exit` fires after `plan_retriever_agent` completes


In [ ]:
# Reset iteration counter before the live run
_sum_costs_iteration = 0

async def run_clean_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_clean_topic()


## 🚫 13b. Run — BLOCKED Topic (Guardrail Still Intercepts)

The topic `"weapons convention"` is still blocked by the 8.4 guardrail.

Notice: `[AUDIT]` never appears — because `accountant_agent` never runs when the workflow is cancelled at the outermost boundary.

This confirms the auditing callback only fires when `sum_costs` is actually called.


In [ ]:
async def run_blocked_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "weapons convention"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_blocked_topic()
